In [1]:
import ConnectionConfig as cc
from delta import DeltaTable
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("factRit")
spark.getActiveSession()

In [3]:
cc.set_connectionProfile("VeloDB")

df_operational_rides = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

#### 2 MAKE DIMENSION TABLES AVAILABLE AS VIEWS


In [ ]:
dimDate = spark.read.format("delta").load("spark-warehouse/date_data_delta")
dimWeather = spark.read.format("delta").load("spark-warehouse/weather_data_delta")
dim_Customer = spark.read.format("delta").load("spark-warehouse/customer_data")
dim_locks = spark.read.format("delta").load("spark-warehouse/slot_data")

In [16]:

dimDate.createOrReplaceTempView("dimDate")
dimWeather.createOrReplaceTempView("dimWeather")
dim_Customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")

In [10]:
df_operational_rides.createOrReplaceTempView("rides_source")


In [23]:
# Create the fact table factRit
# Create the fact table factRit
factRit = spark.sql("""
SELECT
    r.rideId,
    r.starttime as ride_starttime,
    r.endtime as ride_endtime,
    d.dateSK as date_key,
    c.customerSK as customer_key,
    s1.lockSK as start_lock_key,
    s2.lockSK as end_lock_key,
    w.weather_category as weather_key, -- Adjust this to the correct column name based on inspection
    r.vehicleid as vehicle_id,
    (unix_timestamp(r.endtime) - unix_timestamp(r.starttime)) as duration -- Calculate duration in seconds
FROM
    rides_source r
-- Join with Date dimension
JOIN
    dimDate d ON date_format(r.starttime, 'yyyy-MM-dd') = d.calendarDate
-- Join with Customer dimension (SCD Type 2)
JOIN
    dimCustomer c ON r.subscriptionid = c.userId AND r.starttime BETWEEN c.scd_start AND c.scd_end
-- Join with Lock dimension for start lock
JOIN
    dimLocks s1 ON r.startlockid = s1.lockId
-- Join with Lock dimension for end lock
JOIN
    dimLocks s2 ON r.endlockid = s2.lockId
-- Join with Weather dimension
JOIN
    dimWeather w ON date_format(r.starttime, 'yyyy-MM-dd') = w.calendarDate
""")

factRit.show()


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `w`.`calendarDate` cannot be resolved. Did you mean one of the following? [`d`.`calendarDate`, `r`.`endpoint`, `c`.`current`, `r`.`endtime`, `c`.`name`].; line 29 pos 61;
'Project ['r.rideId, 'r.starttime AS ride_starttime#2170, 'r.endtime AS ride_endtime#2171, 'd.dateSK AS date_key#2172, 'c.customerSK AS customer_key#2173, 's1.lockSK AS start_lock_key#2174, 's2.lockSK AS end_lock_key#2175, 'w.weather_category AS weather_key#2176, 'r.vehicleid AS vehicle_id#2177, ('unix_timestamp('r.endtime) - 'unix_timestamp('r.starttime)) AS duration#2178]
+- 'Join Inner, (date_format(starttime#3, yyyy-MM-dd, Some(Europe/Brussels)) = 'w.calendarDate)
   :- Join Inner, (endlockid#8 = cast(lockId#2180 as int))
   :  :- Join Inner, (startlockid#7 = cast(lockId#547 as int))
   :  :  :- Join Inner, ((subscriptionid#6 = userId#517) AND ((starttime#3 >= scd_start#527) AND (starttime#3 <= scd_end#528)))
   :  :  :